In [1]:
import pandas as pd

# Path to the source Excel file
file_path = "D:\WORK - MAIN\PROJECTS\Smart Traffic Data Engineering & Analytics Pipeline\smart_traffic_dataset-v2.xlsx"

# Load both sheets
traffic_df = pd.read_excel(file_path, sheet_name="Traffic_Logs_Raw")
location_df = pd.read_excel(file_path, sheet_name="Location_Lookup")

# Check that the data was loaded correctly
print("Traffic data:")
print(traffic_df.head())

print("\nLocation lookup:")
print(location_df.head())

Traffic data:
    Record_ID           Timestamp  Location_ID Vehicle_Type  Speed_kmh  \
0  TRF-100000 2025-01-01 00:00:00          107          Car       45.8   
1  TRF-100001 2025-01-01 01:00:00          104          Car       65.4   
2  TRF-100002 2025-01-01 02:00:00          105        Truck       64.9   
3  TRF-100003 2025-01-01 03:00:00          107         car        50.1   
4  TRF-100004 2025-01-01 04:00:00          103        Truck       31.7   

   Traffic_Volume Weather_Condition   Lane_Number  
0           567.0             Clear  Express Lane  
1           372.0             clear        Lane 1  
2           169.0             foggy  Express Lane  
3           102.0             Rainy  Express Lane  
4           457.0             Foggy  Express Lane  

Location lookup:
   Location_ID           Zone_Name      Road_Type  Speed_Limit_kmh
0          101       Downtown Core     Expressway               80
1          102    Highway 401 East        Highway              100
2         

In [2]:
print("\nTraffic data info:")
traffic_df.info()

# count missing values
print("\nMissing values in traffic data:")
print(traffic_df.isnull().sum())

print("\nTraffic data columns:")
print(traffic_df.columns.tolist())



Traffic data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Record_ID          12000 non-null  object        
 1   Timestamp          12000 non-null  datetime64[ns]
 2   Location_ID        12000 non-null  int64         
 3   Vehicle_Type       12000 non-null  object        
 4   Speed_kmh          11760 non-null  float64       
 5   Traffic_Volume     11820 non-null  float64       
 6   Weather_Condition  10828 non-null  object        
 7   Lane_Number        12000 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 750.1+ KB

Missing values in traffic data:
Record_ID               0
Timestamp               0
Location_ID             0
Vehicle_Type            0
Speed_kmh             240
Traffic_Volume        180
Weather_Condition    1172
Lane_Number             

In [3]:
print("\nDuplicate rows:")
print(traffic_df.duplicated().sum())

print("\nDuplicate Record_IDs:")
print(traffic_df["Record_ID"].duplicated().sum())


Duplicate rows:
0

Duplicate Record_IDs:
0


In [4]:
print("\nVehicle types:")
print(traffic_df["Vehicle_Type"].value_counts(dropna=False))

print("\nWeather conditions:")
print(traffic_df["Weather_Condition"].value_counts(dropna=False))



Vehicle types:
Vehicle_Type
Truck         2038
Car           2023
Bus           1013
Motorcycle    1006
BUS           1005
TRUCK         1001
Bike           995
car            991
Van            990
bike           938
Name: count, dtype: int64

Weather conditions:
Weather_Condition
Clear     2376
Snowy     1276
foggy     1222
clear     1216
Rainy     1198
Cloudy    1190
Foggy     1178
NaN       1172
RAINY     1172
Name: count, dtype: int64


In [5]:
traffic_clean = traffic_df.copy()

In [6]:
traffic_clean["Vehicle_Type"] = traffic_clean["Vehicle_Type"].str.strip()
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].str.strip()

In [7]:
traffic_clean["Vehicle_Type"] = traffic_clean["Vehicle_Type"].str.lower()
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].str.lower()

In [8]:
print("\nCleaned vehicle types:")
print(traffic_clean["Vehicle_Type"].value_counts(dropna=False))

print("\nCleaned weather conditions:")
print(traffic_clean["Weather_Condition"].value_counts(dropna=False))


Cleaned vehicle types:
Vehicle_Type
truck         3039
car           3014
bus           2018
bike          1933
motorcycle    1006
van            990
Name: count, dtype: int64

Cleaned weather conditions:
Weather_Condition
clear     3592
foggy     2400
rainy     2370
snowy     1276
cloudy    1190
NaN       1172
Name: count, dtype: int64


In [9]:
traffic_clean["Timestamp"] = pd.to_datetime(traffic_clean["Timestamp"], errors="coerce")

In [10]:
print("\nTimestamp data type:")
print(traffic_clean["Timestamp"].dtype)

print("\nInvalid timestamps:")
print(traffic_clean["Timestamp"].isna().sum())


Timestamp data type:
datetime64[ns]

Invalid timestamps:
0


In [11]:
invalid_timestamps = traffic_clean[traffic_clean["Timestamp"].isna()]

print("\nRows with invalid timestamps:")
print(
    traffic_df.loc[
        invalid_timestamps.index,
        ["Record_ID", "Timestamp"]
    ]
)


Rows with invalid timestamps:
Empty DataFrame
Columns: [Record_ID, Timestamp]
Index: []


In [12]:
traffic_clean = traffic_clean.dropna(subset=["Timestamp"])

In [13]:
print("\nRemaining invalid timestamps:")
print(traffic_clean["Timestamp"].isna().sum())

print("\nRows remaining after timestamp cleaning:")
print(len(traffic_clean))


Remaining invalid timestamps:
0

Rows remaining after timestamp cleaning:
12000


In [14]:
print("\nMissing values after timestamp cleaning:")
print(traffic_clean.isna().sum())


Missing values after timestamp cleaning:
Record_ID               0
Timestamp               0
Location_ID             0
Vehicle_Type            0
Speed_kmh             240
Traffic_Volume        180
Weather_Condition    1172
Lane_Number             0
dtype: int64


In [15]:
missing_locations = traffic_clean[traffic_clean["Location_ID"].isna()]

print("\nRows with missing Location_ID:")
print(
    missing_locations[
        ["Record_ID", "Timestamp", "Location_ID", "Traffic_Volume"]
    ]
)


Rows with missing Location_ID:
Empty DataFrame
Columns: [Record_ID, Timestamp, Location_ID, Traffic_Volume]
Index: []


In [16]:
traffic_clean = traffic_clean.dropna(subset=["Location_ID"])

In [17]:
print("\nLocation_ID data type:")
print(traffic_clean["Location_ID"].dtype)

print("\nUnique Location_ID values:")
print(traffic_clean["Location_ID"].value_counts().head(20))


Location_ID data type:
int64

Unique Location_ID values:
Location_ID
105    1541
102    1528
104    1509
101    1502
107    1500
106    1493
103    1465
108    1462
Name: count, dtype: int64


In [18]:
print("\nLocation lookup ID data type:")
print(location_df["Location_ID"].dtype)

print("\nLocation lookup IDs:")
print(location_df["Location_ID"].value_counts().head(20))


Location lookup ID data type:
int64

Location lookup IDs:
Location_ID
101    1
102    1
103    1
104    1
105    1
106    1
107    1
108    1
Name: count, dtype: int64


In [19]:
invalid_location_ids = traffic_clean[
    ~traffic_clean["Location_ID"].isin(location_df["Location_ID"])
]

print("\nTraffic records with no matching Location_ID:")
print(len(invalid_location_ids))


Traffic records with no matching Location_ID:
0


In [20]:
print("\nInvalid Location_ID values:")
print(invalid_location_ids["Location_ID"].value_counts())


Invalid Location_ID values:
Series([], Name: count, dtype: int64)


In [21]:
traffic_clean["Location_ID"] = traffic_clean["Location_ID"].astype("string").str.strip().str.upper()
location_df["Location_ID"] = location_df["Location_ID"].astype("string").str.strip().str.upper()

In [22]:
invalid_location_ids = traffic_clean[
    ~traffic_clean["Location_ID"].isin(location_df["Location_ID"])
]

print("\nUnmatched Location_ID records after standardization:")
print(len(invalid_location_ids))


Unmatched Location_ID records after standardization:
0


In [23]:
traffic_clean = traffic_clean[
    traffic_clean["Location_ID"].isin(location_df["Location_ID"])
]

In [24]:
remaining_invalid_locations = traffic_clean[
    ~traffic_clean["Location_ID"].isin(location_df["Location_ID"])
]

print("\nRemaining unmatched Location_ID records:")
print(len(remaining_invalid_locations))


Remaining unmatched Location_ID records:
0


In [25]:
print("\nMissing Speed values:")
print(traffic_clean["Speed_kmh"].isna().sum())

print("\nSpeed statistics:")
print(traffic_clean["Speed_kmh"].describe())

print("\nRows with missing Speed:")
print(
    traffic_clean[
        traffic_clean["Speed_kmh"].isna()
    ][
        ["Record_ID", "Timestamp", "Location_ID", "Vehicle_Type",
         "Traffic_Volume", "Weather_Condition", "Lane_Number"]
    ].head(10)
)


Missing Speed values:
240

Speed statistics:
count    11760.000000
mean        48.040646
std         17.593776
min          5.000000
25%         35.900000
50%         47.900000
75%         60.000000
max        118.300000
Name: Speed_kmh, dtype: float64

Rows with missing Speed:
      Record_ID           Timestamp Location_ID Vehicle_Type  Traffic_Volume  \
40   TRF-100040 2025-01-02 16:00:00         103   motorcycle             NaN   
88   TRF-100088 2025-01-04 16:00:00         104          bus           464.0   
132  TRF-100132 2025-01-06 12:00:00         101          car           815.0   
141  TRF-100141 2025-01-06 21:00:00         101          car           591.0   
212  TRF-100212 2025-01-09 20:00:00         105          car           493.0   
338  TRF-100338 2025-01-15 02:00:00         104          van           764.0   
379  TRF-100379 2025-01-16 19:00:00         101          van          1015.0   
462  TRF-100462 2025-01-20 06:00:00         106          car           906.0   


In [26]:
print("\nSpeed statistics by Vehicle Type:")
print(
    traffic_clean.groupby("Vehicle_Type")["Speed_kmh"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("\nMissing Speed count by Vehicle Type:")
print(
    traffic_clean.loc[
        traffic_clean["Speed_kmh"].isna(),
        "Vehicle_Type"
    ].value_counts()
)

print("\nSpeed statistics by Location:")
print(
    traffic_clean.groupby("Location_ID")["Speed_kmh"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)


Speed statistics by Vehicle Type:
              count   mean  median  min    max
Vehicle_Type                                  
bike           1893  48.43    48.4  5.0  105.9
bus            1981  47.86    47.3  5.0  118.3
car            2946  47.79    48.2  5.0  105.3
motorcycle      985  47.72    47.9  5.0  104.8
truck          2985  48.21    48.0  5.0  112.6
van             970  48.23    47.6  5.0   99.2

Missing Speed count by Vehicle Type:
Vehicle_Type
car           68
truck         54
bike          40
bus           37
motorcycle    21
van           20
Name: count, dtype: int64

Speed statistics by Location:
             count   mean  median  min    max
Location_ID                                  
101           1466  48.07    48.1  5.0  101.4
102           1493  47.63    47.5  5.0  118.3
103           1435  48.39    48.3  5.0  105.3
104           1473  47.88    47.8  5.0  104.6
105           1514  47.56    47.4  5.0  112.6
106           1468  48.35    48.1  5.0  107.7
107        

In [27]:
traffic_clean["Speed_kmh"] = traffic_clean["Speed_kmh"].fillna(
    traffic_clean["Speed_kmh"].median()
)

In [28]:
print("Missing Speed values after imputation:")
print(traffic_clean["Speed_kmh"].isna().sum())

Missing Speed values after imputation:
0


In [29]:
print("Minimum Speed:", traffic_clean["Speed_kmh"].min())
print("Maximum Speed:", traffic_clean["Speed_kmh"].max())

print("\nRows with Speed <= 0:")
print(
    traffic_clean[
        traffic_clean["Speed_kmh"] <= 0
    ][["Record_ID", "Vehicle_Type", "Location_ID", "Speed_kmh"]]
)

Minimum Speed: 5.0
Maximum Speed: 118.3

Rows with Speed <= 0:
Empty DataFrame
Columns: [Record_ID, Vehicle_Type, Location_ID, Speed_kmh]
Index: []


In [30]:
print("Missing Traffic Volume values:")
print(traffic_clean["Traffic_Volume"].isna().sum())

print("\nTraffic Volume statistics:")
print(
    traffic_clean["Traffic_Volume"].describe()
)

Missing Traffic Volume values:
180

Traffic Volume statistics:
count    11820.000000
mean       559.666667
std        308.530319
min         30.000000
25%        293.000000
50%        560.000000
75%        824.000000
max       1099.000000
Name: Traffic_Volume, dtype: float64


In [31]:
print("\nTraffic Volume by Vehicle Type:")

print(
    traffic_clean.groupby("Vehicle_Type")["Traffic_Volume"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)


Traffic Volume by Vehicle Type:
              count    mean  median   min     max
Vehicle_Type                                     
bike           1903  562.20   561.0  30.0  1099.0
bus            1984  568.11   567.0  30.0  1099.0
car            2977  557.37   560.0  30.0  1099.0
motorcycle      985  549.38   558.0  30.0  1098.0
truck          2993  557.88   557.0  30.0  1099.0
van             978  560.42   552.0  33.0  1098.0


In [32]:
print("\nMissing Traffic Volume by Vehicle Type:")

print(
    traffic_clean.loc[
        traffic_clean["Traffic_Volume"].isna(),
        "Vehicle_Type"
    ].value_counts()
)


Missing Traffic Volume by Vehicle Type:
Vehicle_Type
truck         46
car           37
bus           34
bike          30
motorcycle    21
van           12
Name: count, dtype: int64


In [33]:
traffic_clean["Traffic_Volume"] = traffic_clean["Traffic_Volume"].fillna(
    traffic_clean["Traffic_Volume"].median()
)

In [34]:
print("Missing Traffic Volume values after imputation:")
print(traffic_clean["Traffic_Volume"].isna().sum())

Missing Traffic Volume values after imputation:
0


In [35]:
print("Missing Weather values:")
print(traffic_clean["Weather_Condition"].isna().sum())

print("\nWeather Condition counts:")
print(traffic_clean["Weather_Condition"].value_counts(dropna=False))

Missing Weather values:
1172

Weather Condition counts:
Weather_Condition
clear     3592
foggy     2400
rainy     2370
snowy     1276
cloudy    1190
NaN       1172
Name: count, dtype: int64


In [36]:
print("\nWeather Condition by Location:")

print(
    pd.crosstab(
        traffic_clean["Location_ID"],
        traffic_clean["Weather_Condition"],
        dropna=False
    )
)


Weather Condition by Location:
Weather_Condition  clear  cloudy  foggy  rainy  snowy  NaN
Location_ID                                               
101                  477     154    296    300    148  127
102                  460     145    314    324    144  141
103                  424     137    302    316    141  145
104                  439     167    298    283    177  145
105                  457     150    331    289    182  132
106                  458     155    264    268    178  170
107                  433     145    310    288    154  170
108                  444     137    285    302    152  142


In [37]:
print("\nMissing Weather by Vehicle Type:")

print(
    traffic_clean.loc[
        traffic_clean["Weather_Condition"].isna(),
        "Vehicle_Type"
    ].value_counts()
)


Missing Weather by Vehicle Type:
Vehicle_Type
car           301
truck         280
bike          210
bus           185
van            99
motorcycle     97
Name: count, dtype: int64


In [38]:
print("\nWeather Condition Frequency:")

print(
    traffic_clean["Weather_Condition"]
    .value_counts()
)


Weather Condition Frequency:
Weather_Condition
clear     3592
foggy     2400
rainy     2370
snowy     1276
cloudy    1190
Name: count, dtype: int64


In [39]:
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].fillna(
    traffic_clean["Weather_Condition"].mode()[0]
)

In [40]:
print("Missing Weather values after imputation:")
print(traffic_clean["Weather_Condition"].isna().sum())

Missing Weather values after imputation:
0


In [41]:
print("Missing values in each column:")
print(traffic_clean.isna().sum())

Missing values in each column:
Record_ID            0
Timestamp            0
Location_ID          0
Vehicle_Type         0
Speed_kmh            0
Traffic_Volume       0
Weather_Condition    0
Lane_Number          0
dtype: int64


In [42]:
print("Duplicate rows:")
print(traffic_clean.duplicated().sum())

print("\nDuplicate Record_IDs:")
print(traffic_clean["Record_ID"].duplicated().sum())

Duplicate rows:
0

Duplicate Record_IDs:
0


In [43]:
print("Data types of each column:")
print(traffic_clean.dtypes)

Data types of each column:
Record_ID                    object
Timestamp            datetime64[ns]
Location_ID          string[python]
Vehicle_Type                 object
Speed_kmh                   float64
Traffic_Volume              float64
Weather_Condition            object
Lane_Number                  object
dtype: object


In [44]:
print("Negative Speed values:")
print((traffic_clean["Speed_kmh"] < 0).sum())

print("\nNegative Traffic Volume values:")
print((traffic_clean["Traffic_Volume"] < 0).sum())

Negative Speed values:
0

Negative Traffic Volume values:
0


In [45]:
print("Lane Number statistics:")
print(traffic_clean["Lane_Number"].describe())

print("\nUnique Lane Numbers:")
print(sorted(traffic_clean["Lane_Number"].dropna().unique()))

Lane Number statistics:
count      12000
unique         4
top       Lane 1
freq        3069
Name: Lane_Number, dtype: object

Unique Lane Numbers:
['Express Lane', 'Lane 1', 'Lane 2', 'Lane 3']


In [46]:
print("Missing Lane Number values:")
print(traffic_clean["Lane_Number"].isna().sum())

Missing Lane Number values:
0


In [47]:
print("Record_ID statistics:")
print(traffic_clean["Record_ID"].describe())

print("\nNumber of unique Record_IDs:")
print(traffic_clean["Record_ID"].nunique())

print("\nNumber of rows:")
print(len(traffic_clean))

Record_ID statistics:
count          12000
unique         12000
top       TRF-100000
freq               1
Name: Record_ID, dtype: object

Number of unique Record_IDs:
12000

Number of rows:
12000


In [48]:
print("Original rows:", len(traffic_df))
print("Cleaned rows:", len(traffic_clean))
print("Rows removed:", len(traffic_df) - len(traffic_clean))

Original rows: 12000
Cleaned rows: 12000
Rows removed: 0


In [49]:
print("Vehicle Types:")
print(sorted(traffic_clean["Vehicle_Type"].unique()))

print("\nWeather Conditions:")
print(sorted(traffic_clean["Weather_Condition"].unique()))

Vehicle Types:
['bike', 'bus', 'car', 'motorcycle', 'truck', 'van']

Weather Conditions:
['clear', 'cloudy', 'foggy', 'rainy', 'snowy']


In [50]:
print("Cleaned Dataset Shape:")
print(traffic_clean.shape)

print("\nCleaned Dataset Info:")
traffic_clean.info()

Cleaned Dataset Shape:
(12000, 8)

Cleaned Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Record_ID          12000 non-null  object        
 1   Timestamp          12000 non-null  datetime64[ns]
 2   Location_ID        12000 non-null  string        
 3   Vehicle_Type       12000 non-null  object        
 4   Speed_kmh          12000 non-null  float64       
 5   Traffic_Volume     12000 non-null  float64       
 6   Weather_Condition  12000 non-null  object        
 7   Lane_Number        12000 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(4), string(1)
memory usage: 750.1+ KB


In [51]:
traffic_clean.head(10)

,Record_ID,Timestamp,Location_ID,Vehicle_Type,Speed_kmh,Traffic_Volume,Weather_Condition,Lane_Number
0,TRF-100000,2025-01-01 00:00:00,107,car,45.8,567.0,clear,Express Lane
1,TRF-100001,2025-01-01 01:00:00,104,car,65.4,372.0,clear,Lane 1
2,TRF-100002,2025-01-01 02:00:00,105,truck,64.9,169.0,foggy,Express Lane
3,TRF-100003,2025-01-01 03:00:00,107,car,50.1,102.0,rainy,Express Lane
4,TRF-100004,2025-01-01 04:00:00,103,truck,31.7,457.0,foggy,Express Lane
5,TRF-100005,2025-01-01 05:00:00,108,van,37.8,1012.0,foggy,Express Lane
6,TRF-100006,2025-01-01 06:00:00,105,car,19.6,629.0,clear,Lane 2
7,TRF-100007,2025-01-01 07:00:00,105,car,42.0,126.0,snowy,Lane 3
8,TRF-100008,2025-01-01 08:00:00,107,truck,16.2,507.0,clear,Lane 2
9,TRF-100009,2025-01-01 09:00:00,102,truck,51.9,1051.0,clear,Lane 1


In [52]:
print([name for name in globals() if "location" in name.lower()])

['location_df', 'missing_locations', 'invalid_location_ids', 'remaining_invalid_locations']


In [53]:
print("Traffic Location IDs:")
print(sorted(traffic_clean["Location_ID"].unique()))

print("\nLookup Location IDs:")
print(sorted(location_df["Location_ID"].unique()))

Traffic Location IDs:
['101', '102', '103', '104', '105', '106', '107', '108']

Lookup Location IDs:
['101', '102', '103', '104', '105', '106', '107', '108']


In [54]:
print("Missing values in Location Lookup:")
print(location_df.isna().sum())

print("\nDuplicate Location IDs:")
print(location_df["Location_ID"].duplicated().sum())

Missing values in Location Lookup:
Location_ID        0
Zone_Name          0
Road_Type          0
Speed_Limit_kmh    0
dtype: int64

Duplicate Location IDs:
0


In [55]:
print("Location Lookup Table:")
print(location_df)

Location Lookup Table:
  Location_ID           Zone_Name      Road_Type  Speed_Limit_kmh
0         101       Downtown Core     Expressway               80
1         102    Highway 401 East        Highway              100
2         103  Suburban Ring Road  Main Arterial               60
3         104  Financial District     Commercial               50
4         105  Airport Expressway     Expressway               90
5         106     Tech Park North       Suburban               70
6         107       Harbor Bridge         Bridge               90
7         108     Westside Bypass        Highway              100


In [56]:
print("Location Lookup Data Types:")
print(location_df.dtypes)

Location Lookup Data Types:
Location_ID        string[python]
Zone_Name                  object
Road_Type                  object
Speed_Limit_kmh             int64
dtype: object


In [57]:
print("Missing Location_ID values:")
print(location_df["Location_ID"].isna().sum())

print("\nUnique Location_ID count:")
print(location_df["Location_ID"].nunique())

print("\nTotal rows in lookup table:")
print(len(location_df))

Missing Location_ID values:
0

Unique Location_ID count:
8

Total rows in lookup table:
8


In [58]:
print("Duplicate rows in Location Lookup:")
print(location_df.duplicated().sum())

Duplicate rows in Location Lookup:
0


In [59]:
traffic_enriched = traffic_clean.merge(
    location_df,
    on="Location_ID",
    how="left"
)

In [60]:
print(traffic_enriched.head())

    Record_ID           Timestamp Location_ID Vehicle_Type  Speed_kmh  \
0  TRF-100000 2025-01-01 00:00:00         107          car       45.8   
1  TRF-100001 2025-01-01 01:00:00         104          car       65.4   
2  TRF-100002 2025-01-01 02:00:00         105        truck       64.9   
3  TRF-100003 2025-01-01 03:00:00         107          car       50.1   
4  TRF-100004 2025-01-01 04:00:00         103        truck       31.7   

   Traffic_Volume Weather_Condition   Lane_Number           Zone_Name  \
0           567.0             clear  Express Lane       Harbor Bridge   
1           372.0             clear        Lane 1  Financial District   
2           169.0             foggy  Express Lane  Airport Expressway   
3           102.0             rainy  Express Lane       Harbor Bridge   
4           457.0             foggy  Express Lane  Suburban Ring Road   

       Road_Type  Speed_Limit_kmh  
0         Bridge               90  
1     Commercial               50  
2     Expressw

In [61]:
print("Rows before merge:", len(traffic_clean))
print("Rows after merge:", len(traffic_enriched))

Rows before merge: 12000
Rows after merge: 12000


In [62]:
print("Missing Zone_Name values:")
print(traffic_enriched["Zone_Name"].isna().sum())

Missing Zone_Name values:
0


In [63]:
print("Missing values in location columns:")
print(
    traffic_enriched[
        ["Zone_Name", "Road_Type", "Speed_Limit_kmh"]
    ].isna().sum()
)

Missing values in location columns:
Zone_Name          0
Road_Type          0
Speed_Limit_kmh    0
dtype: int64


In [64]:
print("Rows and columns:")
print(traffic_enriched.shape)

Rows and columns:
(12000, 11)


In [65]:
print("Columns in traffic_enriched:")
print(traffic_enriched.columns.tolist())

Columns in traffic_enriched:
['Record_ID', 'Timestamp', 'Location_ID', 'Vehicle_Type', 'Speed_kmh', 'Traffic_Volume', 'Weather_Condition', 'Lane_Number', 'Zone_Name', 'Road_Type', 'Speed_Limit_kmh']


In [66]:
print("Data types in traffic_enriched:")
print(traffic_enriched.dtypes)

Data types in traffic_enriched:
Record_ID                    object
Timestamp            datetime64[ns]
Location_ID          string[python]
Vehicle_Type                 object
Speed_kmh                   float64
Traffic_Volume              float64
Weather_Condition            object
Lane_Number                  object
Zone_Name                    object
Road_Type                    object
Speed_Limit_kmh               int64
dtype: object


In [67]:
print("Negative Speed values:")
print((traffic_enriched["Speed_kmh"] < 0).sum())

print("\nNegative Traffic Volume values:")
print((traffic_enriched["Traffic_Volume"] < 0).sum())

print("\nNegative Speed Limit values:")
print((traffic_enriched["Speed_Limit_kmh"] < 0).sum())

Negative Speed values:
0

Negative Traffic Volume values:
0

Negative Speed Limit values:
0


In [68]:
print("Duplicate rows after merge:")
print(traffic_enriched.duplicated().sum())

Duplicate rows after merge:
0


In [69]:
print("Duplicate Record_IDs after merge:")
print(traffic_enriched["Record_ID"].duplicated().sum())

Duplicate Record_IDs after merge:
0


In [70]:
print(traffic_enriched.describe())

                 Timestamp     Speed_kmh  Traffic_Volume  Speed_Limit_kmh
count                12000  12000.000000    12000.000000     12000.000000
mean   2025-09-07 23:30:00     48.037833      559.671667        80.059167
min    2025-01-01 00:00:00      5.000000       30.000000        50.000000
25%    2025-05-05 23:45:00     36.200000      296.000000        70.000000
50%    2025-09-07 23:30:00     47.900000      560.000000        90.000000
75%    2026-01-10 23:15:00     59.800000      820.000000        90.000000
max    2026-05-15 23:00:00    118.300000     1099.000000       100.000000
std                    NaN     17.416946      306.207407        17.305484


In [71]:
print(traffic_enriched.dtypes)

Record_ID                    object
Timestamp            datetime64[ns]
Location_ID          string[python]
Vehicle_Type                 object
Speed_kmh                   float64
Traffic_Volume              float64
Weather_Condition            object
Lane_Number                  object
Zone_Name                    object
Road_Type                    object
Speed_Limit_kmh               int64
dtype: object


In [72]:
traffic_enriched.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Record_ID          12000 non-null  object        
 1   Timestamp          12000 non-null  datetime64[ns]
 2   Location_ID        12000 non-null  string        
 3   Vehicle_Type       12000 non-null  object        
 4   Speed_kmh          12000 non-null  float64       
 5   Traffic_Volume     12000 non-null  float64       
 6   Weather_Condition  12000 non-null  object        
 7   Lane_Number        12000 non-null  object        
 8   Zone_Name          12000 non-null  object        
 9   Road_Type          12000 non-null  object        
 10  Speed_Limit_kmh    12000 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(6), string(1)
memory usage: 1.0+ MB


In [73]:
final_traffic = traffic_enriched.copy()

In [74]:
print(final_traffic.head())

    Record_ID           Timestamp Location_ID Vehicle_Type  Speed_kmh  \
0  TRF-100000 2025-01-01 00:00:00         107          car       45.8   
1  TRF-100001 2025-01-01 01:00:00         104          car       65.4   
2  TRF-100002 2025-01-01 02:00:00         105        truck       64.9   
3  TRF-100003 2025-01-01 03:00:00         107          car       50.1   
4  TRF-100004 2025-01-01 04:00:00         103        truck       31.7   

   Traffic_Volume Weather_Condition   Lane_Number           Zone_Name  \
0           567.0             clear  Express Lane       Harbor Bridge   
1           372.0             clear        Lane 1  Financial District   
2           169.0             foggy  Express Lane  Airport Expressway   
3           102.0             rainy  Express Lane       Harbor Bridge   
4           457.0             foggy  Express Lane  Suburban Ring Road   

       Road_Type  Speed_Limit_kmh  
0         Bridge               90  
1     Commercial               50  
2     Expressw

In [75]:
print("Final number of rows:")
print(len(final_traffic))

Final number of rows:
12000


In [76]:
print("Final number of columns:")
print(len(final_traffic.columns))

Final number of columns:
11


In [77]:
print(final_traffic.columns.tolist())

['Record_ID', 'Timestamp', 'Location_ID', 'Vehicle_Type', 'Speed_kmh', 'Traffic_Volume', 'Weather_Condition', 'Lane_Number', 'Zone_Name', 'Road_Type', 'Speed_Limit_kmh']


In [78]:
print("Missing values in final dataset:")
print(final_traffic.isna().sum())

Missing values in final dataset:
Record_ID            0
Timestamp            0
Location_ID          0
Vehicle_Type         0
Speed_kmh            0
Traffic_Volume       0
Weather_Condition    0
Lane_Number          0
Zone_Name            0
Road_Type            0
Speed_Limit_kmh      0
dtype: int64


In [79]:
print("Vehicle Types:")
print(sorted(final_traffic["Vehicle_Type"].unique()))

print("\nWeather Conditions:")
print(sorted(final_traffic["Weather_Condition"].unique()))

print("\nZones:")
print(sorted(final_traffic["Zone_Name"].unique()))

print("\nRoad Types:")
print(sorted(final_traffic["Road_Type"].unique()))

Vehicle Types:
['bike', 'bus', 'car', 'motorcycle', 'truck', 'van']

Weather Conditions:
['clear', 'cloudy', 'foggy', 'rainy', 'snowy']

Zones:
['Airport Expressway', 'Downtown Core', 'Financial District', 'Harbor Bridge', 'Highway 401 East', 'Suburban Ring Road', 'Tech Park North', 'Westside Bypass']

Road Types:
['Bridge', 'Commercial', 'Expressway', 'Highway', 'Main Arterial', 'Suburban']


In [80]:
final_traffic.to_csv("final_traffic_data.csv", index=False)

In [81]:
import os

print("CSV file exists:", os.path.exists("final_traffic_data.csv"))

CSV file exists: True


In [82]:
csv_check = pd.read_csv("final_traffic_data.csv")

print(csv_check.shape)

(12000, 11)


In [83]:
print("Original final dataset rows:", len(final_traffic))
print("CSV rows after reading:", len(csv_check))

Original final dataset rows: 12000
CSV rows after reading: 12000


In [84]:
print("Original final dataset columns:", len(final_traffic.columns))
print("CSV columns after reading:", len(csv_check.columns))

Original final dataset columns: 11
CSV columns after reading: 11


In [85]:
print("Columns are the same:")
print(final_traffic.columns.equals(csv_check.columns))

Columns are the same:
True


In [86]:
print("Missing values in CSV:")
print(csv_check.isna().sum())

Missing values in CSV:
Record_ID            0
Timestamp            0
Location_ID          0
Vehicle_Type         0
Speed_kmh            0
Traffic_Volume       0
Weather_Condition    0
Lane_Number          0
Zone_Name            0
Road_Type            0
Speed_Limit_kmh      0
dtype: int64


In [87]:
print("Data types after reading CSV:")
print(csv_check.dtypes)

Data types after reading CSV:
Record_ID             object
Timestamp             object
Location_ID            int64
Vehicle_Type          object
Speed_kmh            float64
Traffic_Volume       float64
Weather_Condition     object
Lane_Number           object
Zone_Name             object
Road_Type             object
Speed_Limit_kmh        int64
dtype: object


In [88]:
print("Duplicate rows in CSV:")
print(csv_check.duplicated().sum())

Duplicate rows in CSV:
0


In [89]:
print("Duplicate Record_IDs in CSV:")
print(csv_check["Record_ID"].duplicated().sum())

Duplicate Record_IDs in CSV:
0


In [90]:
print("Rows:", len(csv_check))
print("Columns:", len(csv_check.columns))

print("\nMissing values:")
print(csv_check.isna().sum().sum())

print("\nDuplicate rows:")
print(csv_check.duplicated().sum())

print("\nDuplicate Record_IDs:")
print(csv_check["Record_ID"].duplicated().sum())

Rows: 12000
Columns: 11

Missing values:
0

Duplicate rows:
0

Duplicate Record_IDs:
0


In [91]:
print("Shape:", csv_check.shape)

print("\nColumn names:")
print(csv_check.columns.tolist())

print("\nData types:")
print(csv_check.dtypes)

Shape: (12000, 11)

Column names:
['Record_ID', 'Timestamp', 'Location_ID', 'Vehicle_Type', 'Speed_kmh', 'Traffic_Volume', 'Weather_Condition', 'Lane_Number', 'Zone_Name', 'Road_Type', 'Speed_Limit_kmh']

Data types:
Record_ID             object
Timestamp             object
Location_ID            int64
Vehicle_Type          object
Speed_kmh            float64
Traffic_Volume       float64
Weather_Condition     object
Lane_Number           object
Zone_Name             object
Road_Type             object
Speed_Limit_kmh        int64
dtype: object


In [92]:
print(csv_check.head(10))

    Record_ID            Timestamp  Location_ID Vehicle_Type  Speed_kmh  \
0  TRF-100000  2025-01-01 00:00:00          107          car       45.8   
1  TRF-100001  2025-01-01 01:00:00          104          car       65.4   
2  TRF-100002  2025-01-01 02:00:00          105        truck       64.9   
3  TRF-100003  2025-01-01 03:00:00          107          car       50.1   
4  TRF-100004  2025-01-01 04:00:00          103        truck       31.7   
5  TRF-100005  2025-01-01 05:00:00          108          van       37.8   
6  TRF-100006  2025-01-01 06:00:00          105          car       19.6   
7  TRF-100007  2025-01-01 07:00:00          105          car       42.0   
8  TRF-100008  2025-01-01 08:00:00          107        truck       16.2   
9  TRF-100009  2025-01-01 09:00:00          102        truck       51.9   

   Traffic_Volume Weather_Condition   Lane_Number           Zone_Name  \
0           567.0             clear  Express Lane       Harbor Bridge   
1           372.0           

In [93]:
import os

file_size = os.path.getsize("final_traffic_data.csv")

print("CSV file size:", file_size, "bytes")

CSV file size: 1160526 bytes


In [94]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
C:\Users\Daksh Bhardwaj\anaconda3


In [95]:
print("Final dataset shape:", final_traffic.shape)

print("\nTotal missing values:", final_traffic.isna().sum().sum())

print("Duplicate rows:", final_traffic.duplicated().sum())

print("Duplicate Record_IDs:",
      final_traffic["Record_ID"].duplicated().sum())

Final dataset shape: (12000, 11)

Total missing values: 0
Duplicate rows: 0
Duplicate Record_IDs: 0


In [96]:
final_traffic.to_excel("final_traffic_data.xlsx", index=False)

In [97]:
print("Excel file exists:", os.path.exists("final_traffic_data.xlsx"))

Excel file exists: True


In [98]:
excel_check = pd.read_excel("final_traffic_data.xlsx")

print("Excel data shape:", excel_check.shape)

Excel data shape: (12000, 11)


In [99]:
print("Final dataset rows:", len(final_traffic))
print("Excel rows:", len(excel_check))

Final dataset rows: 12000
Excel rows: 12000


In [100]:
print("Columns are the same:")
print(final_traffic.columns.equals(excel_check.columns))

Columns are the same:
True


In [101]:
print("Total missing values in Excel:")
print(excel_check.isna().sum().sum())

Total missing values in Excel:
0


In [102]:
print("Duplicate rows in Excel:")
print(excel_check.duplicated().sum())

print("\nDuplicate Record_IDs in Excel:")
print(excel_check["Record_ID"].duplicated().sum())

Duplicate rows in Excel:
0

Duplicate Record_IDs in Excel:
0


In [103]:
print("Excel shape:", excel_check.shape)

print("\nExcel data types:")
print(excel_check.dtypes)

Excel shape: (12000, 11)

Excel data types:
Record_ID                    object
Timestamp            datetime64[ns]
Location_ID                   int64
Vehicle_Type                 object
Speed_kmh                   float64
Traffic_Volume                int64
Weather_Condition            object
Lane_Number                  object
Zone_Name                    object
Road_Type                    object
Speed_Limit_kmh               int64
dtype: object


In [104]:
print("CSV exists:", os.path.exists("final_traffic_data.csv"))
print("Excel exists:", os.path.exists("final_traffic_data.xlsx"))

CSV exists: True
Excel exists: True


In [105]:
print("CSV shape:", csv_check.shape)
print("Excel shape:", excel_check.shape)

print("\nCSV columns = Excel columns:")
print(csv_check.columns.equals(excel_check.columns))

CSV shape: (12000, 11)
Excel shape: (12000, 11)

CSV columns = Excel columns:
True


In [106]:
print("===== FINAL OUTPUT CHECK =====")

print("\nCSV:")
print("Shape:", csv_check.shape)
print("Missing values:", csv_check.isna().sum().sum())
print("Duplicate rows:", csv_check.duplicated().sum())
print("Duplicate Record_IDs:", csv_check["Record_ID"].duplicated().sum())

print("\nEXCEL:")
print("Shape:", excel_check.shape)
print("Missing values:", excel_check.isna().sum().sum())
print("Duplicate rows:", excel_check.duplicated().sum())
print("Duplicate Record_IDs:", excel_check["Record_ID"].duplicated().sum())

print("\nCSV vs Excel:")
print("Same columns:", csv_check.columns.equals(excel_check.columns))

===== FINAL OUTPUT CHECK =====

CSV:
Shape: (12000, 11)
Missing values: 0
Duplicate rows: 0
Duplicate Record_IDs: 0

EXCEL:
Shape: (12000, 11)
Missing values: 0
Duplicate rows: 0
Duplicate Record_IDs: 0

CSV vs Excel:
Same columns: True


In [140]:
print("Final columns and data types:")
print(final_traffic.dtypes)

Final columns and data types:
Record_ID                    object
Timestamp            datetime64[ns]
Location_ID          string[python]
Vehicle_Type                 object
Speed_kmh                   float64
Traffic_Volume              float64
Weather_Condition            object
Lane_Number                  object
Zone_Name                    object
Road_Type                    object
Speed_Limit_kmh               int64
dtype: object


In [141]:
location_df.to_csv("locations.csv", index=False)

In [142]:
import os

print(os.path.exists("locations.csv"))

True


In [143]:
print(os.path.abspath("locations.csv"))

C:\Users\Daksh Bhardwaj\anaconda3\locations.csv


In [144]:
final_traffic.to_csv("final_traffic_data.csv", index=False)

In [145]:
import os

print(os.path.abspath("final_traffic_data.csv"))

C:\Users\Daksh Bhardwaj\anaconda3\final_traffic_data.csv


In [146]:
traffic_to_load.to_csv("traffic_logs.csv", index=False)

In [147]:
print(os.path.abspath("traffic_logs.csv"))

C:\Users\Daksh Bhardwaj\anaconda3\traffic_logs.csv
